In [1]:
import pandas as pd
import requests
import time
from tqdm import tqdm

In [2]:
file = pd.read_csv('/content/landslide_with_slope (1).csv')
print(file.shape)
file.head()

(2059, 15)


,Sl.No.,Slide_No,State,District,Slide_Name,NH_SH_Location,Latitude,Longitude,Material_Involved,Movement_Type,Material_Involved_Clean,Movement_Type_Clean,has_history_date,event_year,slope_deg
0,15,ASM/HLKD/83D10/2016/60,Assam,Hailakandi,Ramchandi (Koyabasti),Ramchandi area,24.515944,92.540889,Earth,Slide,Earth,Slide,1,2016.0,0.26
1,14,ASM/HLKD/83D10/2016/59,Assam,Hailakandi,Lalcherra Slide (Rabinala),Near Patakhouri Durga Mandir\nUnder Bilaipur F...,24.511306,92.693028,Debris,Slide,Debris,Slide,1,2016.0,1.65
2,32,ASM/HLK/83D10/2020/003,Assam,Hailakandi,Mohanpur Grant Slide 1,"Mohanpur Grant, Hailakandi\nCircle.",24.703640,92.646810,Debris,Slide,Debris,Slide,1,2020.0,1.03
3,34,ASM/HLK/83D10/2020/004,Assam,Hailakandi,Mohanpur Grant Slide 2,"Mohanpur Grant, Hailakandi\nCircle.",24.706560,92.650080,Debris,Slide,Debris,Slide,1,2020.0,0.93
4,35,AS/KAR/83D06/2009-10/AS-2,Assam,Karimganj,Rongpur slide,Rongpur Village,24.710833,92.481167,Debris,Slide,Debris,Slide,1,2010.0,0.67


In [3]:
test_row = file[file['event_year'].notna()].iloc[0]
print(test_row[['Latitude', 'Longitude', 'event_year']])

Latitude      24.515944
Longitude     92.540889
event_year       2016.0
Name: 0, dtype: object


In [4]:
def get_monsoon_avg_rainfall(lat, lon, year, retries=3, timeout=20):
    """
    Fetches historical monsoon-season (June-Sept) rainfall for a specific
    location and year using Open-Meteo's Archive API.

    Used for Group A rows (landslide records where the event_year is known).
    Returns the average daily rainfall (mm) across the monsoon season for
    that year, or None if the API call fails after all retries.
    """
    url = "https://archive-api.open-meteo.com/v1/archive"
    request_detail = {
        "latitude": lat,
        "longitude": lon,
        "start_date": f"{int(year)}-06-01",
        "end_date": f"{int(year)}-09-30",
        "daily": "precipitation_sum",
        "timezone": "Asia/Kolkata"
    }
    for attempt in range(retries):
        try:
            response = requests.get(url, params=request_detail, timeout=timeout)
            response.raise_for_status()
            data = response.json()
            values = data.get("daily", {}).get("precipitation_sum", [])
            values = [v for v in values if v is not None]
            if values:
                return sum(values) / len(values)  # average daily rainfall over monsoon season
            return None
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt+1} failed: {e}")
    return None

In [5]:
lat = test_row['Latitude']
lon = test_row['Longitude']
year = test_row['event_year']

result = get_monsoon_avg_rainfall(lat, lon, year)
print("Average monsoon rainfall (mm/day):", result)

Average monsoon rainfall (mm/day): 12.18360655737705


In [6]:
test_row_b = file[file['event_year'].isna()].iloc[0]
print(test_row_b[['Latitude', 'Longitude', 'District', 'event_year']])

Latitude           26.210833
Longitude          91.689444
District      Kamrup (Metro)
event_year               NaN
Name: 280, dtype: object


In [7]:
def get_longterm_avg_rainfall(lat, lon, years=range(2015, 2025), retries=3, timeout=20):
    """
    Estimates typical monsoon-season rainfall for a location when the
    exact landslide year is unknown (Group B rows).

    Calls get_monsoon_avg_rainfall() once per year across a 10-year window
    (2015-2024 by default), then averages those yearly results into one
    long-term rainfall value for this lat/lon.
    """
    yearly_averages = []
    for year in years:
        avg = get_monsoon_avg_rainfall(lat, lon, year, retries=retries, timeout=timeout)
        if avg is not None:
            yearly_averages.append(avg)
        time.sleep(0.3)  # small delay between calls to avoid rate-limiting
    if yearly_averages:
        return sum(yearly_averages) / len(yearly_averages)
    return None

In [8]:
lat_b = test_row_b['Latitude']
lon_b = test_row_b['Longitude']

result_b = get_longterm_avg_rainfall(lat_b, lon_b)
print("Long-term average monsoon rainfall (mm/day):", result_b)

Long-term average monsoon rainfall (mm/day): 14.712459016393442


In [9]:
district_centroids = file.groupby('District')[['Latitude', 'Longitude']].mean().reset_index()
print(district_centroids)

               District   Latitude  Longitude
0                 Anjaw  28.137573  96.505363
1            Bongaigaon  26.384033  90.541400
2                Cachar  24.886506  92.776695
3             Changlang  27.244211  95.960602
4         Dibang Valley  28.644035  95.861470
5            Dima Hasao  25.234706  93.088080
6           East Kameng  27.368379  93.042892
7            East Siang  28.166639  95.291810
8              Goalpara  26.088333  90.499333
9            Hailakandi  24.523020  92.624005
10               Jorhat  26.565500  94.357500
11                Kamle  27.787075  94.082994
12               Kamrup  26.129427  91.838540
13       Kamrup (Metro)  26.172621  91.787140
14       Kamrup (Rural)  25.872500  91.317500
15        Karbi Anglong  26.197971  93.471588
16            Karimganj  24.683777  92.423976
17            Kra Daadi  27.726188  93.574381
18         Kurung Kumey  27.847263  93.477164
19            Lepa Rada  27.968361  94.652935
20                Lohit  27.934822

In [10]:
# Group B step 1: Calculate one long-term (2015-2024) average monsoon rainfall
# value per district (41 districts), instead of querying each of the 1508
# individual rows separately. Results are stored in a dictionary for lookup.
district_rainfall = {}
for _, row in tqdm(district_centroids.iterrows(), total=len(district_centroids)):
    district = row['District']
    avg_rain = get_longterm_avg_rainfall(row['Latitude'], row['Longitude'])
    district_rainfall[district] = avg_rain

100%|██████████| 41/41 [06:21<00:00,  9.31s/it]


In [11]:
print(district_rainfall)

{'Anjaw': 41.49426229508197, 'Bongaigaon': 22.98639344262295, 'Cachar': 16.40983606557377, 'Changlang': 14.193934426229509, 'Dibang Valley': 15.507131147540983, 'Dima Hasao': 8.835655737704917, 'East Kameng': 15.144016393442623, 'East Siang': 21.948934426229506, 'Goalpara': 16.847704918032786, 'Hailakandi': 12.933852459016393, 'Jorhat': 11.91311475409836, 'Kamle': 15.065081967213114, 'Kamrup': 13.51032786885246, 'Kamrup (Metro)': 14.779918032786885, 'Kamrup (Rural)': 15.055737704918034, 'Karbi Anglong': 12.949672131147542, 'Karimganj': 14.749918032786885, 'Kra Daadi': 21.272950819672133, 'Kurung Kumey': 18.742295081967214, 'Lepa Rada': 19.09811475409836, 'Lohit': 22.200245901639345, 'Longding': 13.756967213114754, 'Lower Dibang Valley': 18.489918032786886, 'Lower Siang': 18.53622950819672, 'Lower Subansiri': 16.439344262295084, 'Morigaon': 15.437786885245902, 'Nagaon': 13.9172131147541, 'Pakke Kessang': 13.991475409836065, 'Papum Pare': 15.31327868852459, 'Shi-Yomi': 22.715655737704918

In [12]:
# Group B step 2: Assign the district-level rainfall averages (calculated in the
# previous cell) to all rows that don't have a known event_year. Each row looks
# up its district's average rainfall from the district_rainfall dictionary.
def fill_rainfall_group_b(row):
    return district_rainfall.get(row['District'], None)

mark_group_a = file['event_year'].notna()
mark_group_b = file['event_year'].isna()
file.loc[mark_group_b, 'rainfall_mm'] = file.loc[mark_group_b, 'District'].map(district_rainfall)

In [13]:
print(file.loc[mark_group_b, 'rainfall_mm'].isna().sum())  # should print 0
print(file[file['event_year'].isna()][['District', 'rainfall_mm']].head(10))  # eyeball check

0
           District  rainfall_mm
280  Kamrup (Metro)    14.779918
401      Hailakandi    12.933852
402      Hailakandi    12.933852
403      Hailakandi    12.933852
404      Hailakandi    12.933852
405      Hailakandi    12.933852
406       Karimganj    14.749918
407      West Siang    18.935410
408     Upper Siang    17.539918
409     Upper Siang    17.539918


In [14]:
current_year = 2026  # today's year, monsoon season may not be complete

for idx in tqdm(file[mark_group_a].index):
    lat = file.loc[idx, 'Latitude']
    lon = file.loc[idx, 'Longitude']
    year = file.loc[idx, 'event_year']
    district = file.loc[idx, 'District']

    if year >= current_year:
        # Can't fetch future/incomplete-season data — use district average instead
        rainfall = district_rainfall.get(district, None)
    else:
        rainfall = get_monsoon_avg_rainfall(lat, lon, year)

    file.loc[idx, 'rainfall_mm'] = rainfall
    time.sleep(0.3)

 23%|██▎       | 128/551 [01:59<06:35,  1.07it/s]

Attempt 1 failed: HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Read timed out. (read timeout=20)


 58%|█████▊    | 319/551 [05:17<03:35,  1.08it/s]

Attempt 1 failed: HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Read timed out. (read timeout=20)


100%|██████████| 551/551 [09:12<00:00,  1.00s/it]


In [15]:
print(file['rainfall_mm'].isna().sum())  # how many rows still missing rainfall - should be 0 or very close
print(file.shape)  # should be (2059, 16)

0
(2059, 16)


In [16]:
file.to_csv('landslide_with_rainfall.csv', index=False)

## Soil Moisture Feature

Same approach as rainfall: for Group A (known event_year), fetch the
exact year's monsoon-season soil moisture from Open-Meteo. For Group B
(unknown event_year), use the district-level long-term (2015-2024)
average. Feature: `soil_moisture_mm` (0-7cm depth, volumetric,
monsoon-season average) — surface-level moisture that correlates
directly with rainfall-triggered landslides.

In [17]:
# ===== SOIL MOISTURE FEATURE =====
# Same Group A/B strategy as rainfall: exact-year fetch for known dates,
# district-level long-term average for unknown dates.

def get_monsoon_avg_soil_moisture(lat, lon, year, retries=3, timeout=20):
    """
    Fetches historical monsoon-season (June-Sept) soil moisture (0-7cm depth)
    for a specific location and year using Open-Meteo's Archive API.
    Returns the average volumetric soil moisture (m3/m3) across the
    monsoon season, or None if the API call fails after all retries.
    """
    url = "https://archive-api.open-meteo.com/v1/archive"
    request_detail = {
        "latitude": lat,
        "longitude": lon,
        "start_date": f"{int(year)}-06-01",
        "end_date": f"{int(year)}-09-30",
        "hourly": "soil_moisture_0_to_7cm",
        "timezone": "Asia/Kolkata"
    }
    for attempt in range(retries):
        try:
            response = requests.get(url, params=request_detail, timeout=timeout)
            response.raise_for_status()
            data = response.json()
            values = data.get("hourly", {}).get("soil_moisture_0_to_7cm", [])
            values = [v for v in values if v is not None]
            if values:
                return sum(values) / len(values)
            return None
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(1)
    return None


def get_longterm_avg_soil_moisture(lat, lon, years=range(2015, 2025), retries=3, timeout=20):
    """
    Estimates typical monsoon-season soil moisture for a location when the
    exact landslide year is unknown (Group B rows).
    """
    yearly_averages = []
    for year in years:
        avg = get_monsoon_avg_soil_moisture(lat, lon, year, retries=retries, timeout=timeout)
        if avg is not None:
            yearly_averages.append(avg)
        time.sleep(0.3)
    if yearly_averages:
        return sum(yearly_averages) / len(yearly_averages)
    return None

In [18]:
# Group B step 1: one long-term (2015-2024) average soil moisture per district
# (reusing the same district_centroids computed earlier for rainfall)
district_soil_moisture = {}
for _, row in tqdm(district_centroids.iterrows(), total=len(district_centroids)):
    district = row['District']
    avg_sm = get_longterm_avg_soil_moisture(row['Latitude'], row['Longitude'])
    district_soil_moisture[district] = avg_sm

 46%|████▋     | 19/41 [03:01<03:28,  9.48s/it]

Attempt 1 failed: HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Read timed out. (read timeout=20)


100%|██████████| 41/41 [06:50<00:00, 10.02s/it]


In [19]:
# Group B step 2: assign district-level soil moisture to unknown-year rows
file.loc[mark_group_b, 'soil_moisture_mm'] = file.loc[mark_group_b, 'District'].map(district_soil_moisture)

print(file.loc[mark_group_b, 'soil_moisture_mm'].isna().sum())  # should print 0

0


In [20]:
# Group A: exact year fetch, with fallback to district average
for idx in tqdm(file[mark_group_a].index):
    lat = file.loc[idx, 'Latitude']
    lon = file.loc[idx, 'Longitude']
    year = file.loc[idx, 'event_year']
    district = file.loc[idx, 'District']

    if year >= current_year:
        soil_moisture = district_soil_moisture.get(district, None)
    else:
        soil_moisture = get_monsoon_avg_soil_moisture(lat, lon, year)
        if soil_moisture is None:  # fallback if all retries fail
            soil_moisture = district_soil_moisture.get(district, None)

    file.loc[idx, 'soil_moisture_mm'] = soil_moisture
    time.sleep(0.3)

100%|██████████| 551/551 [08:38<00:00,  1.06it/s]


In [21]:
print(file['soil_moisture_mm'].isna().sum())  # should be 0 or very close
print(file.shape)  # should be (2059, 17)

0
(2059, 17)


In [31]:
file.to_csv('landslide_with_soil_moisture.csv', index=False)
print("Saved:", file.shape)

Saved: (2059, 18)


## Negative Sample Generation

The GSI dataset only contains locations where landslides actually occurred
(positive samples). To train a classifier that can distinguish risky from
safe terrain, we generate synthetic "no landslide" points using district
centroids with random jitter, filtered to stay at least 5km away from any
real landslide location (and 2km from other generated points, to avoid
clustering). We then compute the same features (slope, rainfall, soil
moisture) for these points as we did for the positive samples, and label
them `0` for training.

In [24]:
import math
ELEVATION_BASE_URL = "https://api.open-meteo.com/v1/elevation"

def get_elevation(lat: float, lon: float, retries: int = 3, timeout: int = 30):
    """
    Fetches elevation (in meters) for a given lat/lon using Open-Meteo's
    Elevation API. Retries on failure, returns None if all attempts fail.
    """
    request_detail = {"latitude": lat, "longitude": lon}
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(ELEVATION_BASE_URL, params=request_detail, timeout=timeout)
            response.raise_for_status()
            data = response.json()
            return data["elevation"][0] if data.get("elevation") else None
        except requests.exceptions.RequestException as e:
            pass
    return None

def estimate_slope_degrees(lat: float, lon: float, offset: float = 0.01):
    """
    Estimates terrain slope (in degrees) at a location by comparing its
    elevation to a nearby point (offset by ~1.1km north) and calculating
    the angle of elevation change over that distance.
    """
    elev_center = get_elevation(lat, lon)
    elev_offset = get_elevation(lat + offset, lon)
    if elev_center is None or elev_offset is None:
        return None
    horizontal_distance_m = offset * 111320
    vertical_diff_m = abs(elev_offset - elev_center)
    if horizontal_distance_m == 0:
        return 0.0
    slope_rad = math.atan(vertical_diff_m / horizontal_distance_m)
    return round(math.degrees(slope_rad), 2)

In [25]:
# ===== NEGATIVE SAMPLE GENERATION =====
# No external dataset needed — we generate candidate "no landslide" points
# ourselves, using the district_centroids and actual positive-point locations
# we already have in this notebook.

import numpy as np
from math import radians, sin, cos, sqrt, atan2

def haversine_km(lat1, lon1, lat2, lon2):
    """Distance in km between two lat/lon points."""
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def is_far_from_all(lat, lon, positive_lats, positive_lons, existing_negatives, min_km=5, min_km_neg=2):
    """Check candidate point isn't too close to any real landslide point,
    and isn't too close to any negative point already generated."""
    for plat, plon in zip(positive_lats, positive_lons):
        if haversine_km(lat, lon, plat, plon) < min_km:
            return False
    for neg in existing_negatives:
        if haversine_km(lat, lon, neg['Latitude'], neg['Longitude']) < min_km_neg:
            return False
    return True

# Actual positive-point coordinates, for distance-filtering
positive_lats = file['Latitude'].values
positive_lons = file['Longitude'].values

points_per_district = 50   # 41 districts x 50 = ~2050 negative points
jitter_deg = 0.25          # how far to scatter around each district centroid
max_attempts_per_point = 20

negative_points = []

for _, row in district_centroids.iterrows():
    district = row['District']
    center_lat = row['Latitude']
    center_lon = row['Longitude']

    collected = 0
    attempts = 0
    while collected < points_per_district and attempts < points_per_district * max_attempts_per_point:
        attempts += 1
        cand_lat = center_lat + np.random.uniform(-jitter_deg, jitter_deg)
        cand_lon = center_lon + np.random.uniform(-jitter_deg, jitter_deg)

        if is_far_from_all(cand_lat, cand_lon, positive_lats, positive_lons, negative_points, min_km=5, min_km_neg=2):
            negative_points.append({
                'District': district,
                'Latitude': cand_lat,
                'Longitude': cand_lon,
                'label': 0
            })
            collected += 1

negative_data_file = pd.DataFrame(negative_points)
print(negative_data_file.shape)
print(negative_data_file.head())

(2050, 4)
  District   Latitude  Longitude  label
0    Anjaw  28.281284  96.273166      0
1    Anjaw  28.311076  96.704463      0
2    Anjaw  28.252727  96.678112      0
3    Anjaw  28.049689  96.717083      0
4    Anjaw  28.118791  96.608711      0


In [26]:
# --- Slope ---
slopes_neg = []
for idx, row in tqdm(negative_data_file.iterrows(), total=len(negative_data_file)):
    slope = estimate_slope_degrees(row["Latitude"], row["Longitude"])
    slopes_neg.append(slope)
    time.sleep(0.3)
negative_data_file["slope_deg"] = slopes_neg

negative_data_file["slope_deg"] = negative_data_file.groupby("District")["slope_deg"].transform(
    lambda x: x.fillna(x.median())
)
negative_data_file["slope_deg"] = negative_data_file["slope_deg"].fillna(negative_data_file["slope_deg"].median())

print("Missing slope:", negative_data_file["slope_deg"].isna().sum())

100%|██████████| 2050/2050 [56:40<00:00,  1.66s/it]

Missing slope: 0


In [27]:
# --- Rainfall & Soil moisture (district-average, instant lookup) ---
negative_data_file["rainfall_mm"] = negative_data_file["District"].map(district_rainfall)
negative_data_file["soil_moisture_mm"] = negative_data_file["District"].map(district_soil_moisture)

# add small variation so same-district negatives aren't identical
noise_factor = 0.1
negative_data_file['rainfall_mm'] = negative_data_file['rainfall_mm'] * (1 + np.random.uniform(-noise_factor, noise_factor, len(negative_data_file)))
negative_data_file['soil_moisture_mm'] = negative_data_file['soil_moisture_mm'] * (1 + np.random.uniform(-noise_factor, noise_factor, len(negative_data_file)))

print("Missing rainfall:", negative_data_file["rainfall_mm"].isna().sum())
print("Missing soil moisture:", negative_data_file["soil_moisture_mm"].isna().sum())

Missing rainfall: 0
Missing soil moisture: 0


In [32]:
print(negative_data_file.columns.tolist())
print(negative_data_file.isna().sum())
negative_data_file.to_csv('negative_with_all_features.csv', index=False)
print("All features saved. Shape:", negative_data_file.shape)

['District', 'Latitude', 'Longitude', 'label', 'slope_deg', 'rainfall_mm', 'soil_moisture_mm']
District            0
Latitude            0
Longitude           0
label               0
slope_deg           0
rainfall_mm         0
soil_moisture_mm    0
dtype: int64
All features saved. Shape: (2050, 7)


In [34]:
#merging of positive and negative data
file['label'] = 1

district_to_state = file.drop_duplicates('District').set_index('District')['State'].to_dict()

negative_data_file['State'] = negative_data_file['District'].map(district_to_state)

positive_final_data = file[['State','District', 'Latitude', 'Longitude', 'slope_deg', 'rainfall_mm', 'soil_moisture_mm', 'label']]
negative_final_data = negative_data_file[['State','District', 'Latitude', 'Longitude', 'slope_deg', 'rainfall_mm', 'soil_moisture_mm', 'label']]

training_data = pd.concat([positive_final_data, negative_final_data], ignore_index=True)
print(training_data.shape)
print(training_data['label'].value_counts())
print(training_data['State'].isna().sum())

(4109, 8)
label
1    2059
0    2050
Name: count, dtype: int64
0


In [35]:
training_data.to_csv('training_data_final.csv', index=False)
print("Training data saved:", training_data.shape)

Training data saved: (4109, 8)
